Homework, not run on the day.

# Chen–Pelger–Zhu (2024) — conceptual sketch

**Paper:** Chen, L., Pelger, M., & Zhu, J. (2024). *Deep Learning in Asset Pricing.* *Management Science*, 70(2), 714–750.

This is **not** a replication (no adversarial SDF estimation, no macro-state network).
It is a classroom sketch of their *idea*: after a Gu-style predictive model, ask what changes if we care about an **economic** objective, not only forecast MSE.

**Companion:** `sessions/04_chen_pelger_zhu.md` · `slides/04_chen_wrap.md`

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2024)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1. Tiny return panel (Gu-style toy)

Planted signal + noise. We will compare:

1. **MSE ranking** — sort by \(\hat r\) from Ridge;
2. **Economically tilted ranking** — shrink forecasts toward a long–short book that looks more “SDF-like” (toy penalty).

In [ ]:
N, T, P = 120, 120, 8
try:
    dates = pd.date_range("2012-01-31", periods=T, freq="ME")
except ValueError:
    dates = pd.date_range("2012-01-31", periods=T, freq="M")

rows = []
for dt in dates:
    X = rng.normal(size=(N, P))
    Xr = (pd.DataFrame(X).rank(axis=0, pct=True).to_numpy() - 0.5) * 2
    f = 0.015 * Xr[:, 0] + 0.01 * (Xr[:, 1] * Xr[:, 2])
    r = f + rng.normal(scale=0.05, size=N)
    df = pd.DataFrame(Xr, columns=[f"x{j}" for j in range(P)])
    df["date"] = dt
    df["stock"] = np.arange(N)
    df["ret"] = r
    rows.append(df)

panel = pd.concat(rows, ignore_index=True)
features = [c for c in panel.columns if c.startswith("x")]
train = panel[panel["date"] <= "2018-12-31"]
test = panel[panel["date"] > "2018-12-31"]
X_train, y_train = train[features].to_numpy(), train["ret"].to_numpy()
X_test = test[features].to_numpy()
print(train["date"].nunique(), "train months;", test["date"].nunique(), "test months")

## 2. Baseline predictive model (MSE spirit)

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=5.0)),
])
model.fit(X_train, y_train)
test = test.copy()
test["r_hat_mse"] = model.predict(X_test)

## 3. Toy “economic” adjustment

Chen et al. encode no-arbitrage / SDF structure in the **training objective**.

Here we only illustrate a **post-hoc tilt** (pedagogical stand-in):

- each month, form a crude long–short from \(\hat r\);
- shrink individual forecasts toward the portfolio’s demeaned signal so extreme cross-sectional fits are less rewarded unless they help the book.

This is **not** their estimator — it makes the debate concrete before students open the paper.

In [ ]:
def tilt_forecasts(df, col="r_hat_mse", q=0.2, shrink=0.35):
    out = []
    for dt, g in df.groupby("date"):
        g = g.copy()
        lo, hi = g[col].quantile(q), g[col].quantile(1 - q)
        # portfolio direction: +1 long candidates, -1 short, 0 else
        side = np.where(g[col] >= hi, 1.0, np.where(g[col] <= lo, -1.0, 0.0))
        # tilt: pull forecasts toward a mean long-short residual
        ls_level = g.loc[side == 1, col].mean() - g.loc[side == -1, col].mean()
        target = side * (ls_level / 2.0)
        g["r_hat_econ"] = (1 - shrink) * g[col] + shrink * target
        out.append(g)
    return pd.concat(out, ignore_index=True)

test_t = tilt_forecasts(test)

## 4. Compare long–short performance

In [ ]:
def long_short(df, score_col, q=0.1):
    rows = []
    for dt, g in df.groupby("date"):
        lo, hi = g[score_col].quantile(q), g[score_col].quantile(1 - q)
        long = g.loc[g[score_col] >= hi, "ret"].mean()
        short = g.loc[g[score_col] <= lo, "ret"].mean()
        rows.append({"date": dt, "ls": long - short})
    return pd.DataFrame(rows).set_index("date").sort_index()

def sharpe(x):
    return np.sqrt(12) * x.mean() / x.std(ddof=1)

ls_mse = long_short(test_t, "r_hat_mse")
ls_econ = long_short(test_t, "r_hat_econ")

summary = pd.DataFrame([
    {"rule": "MSE forecasts", "mean_m": ls_mse["ls"].mean(), "sharpe_ann": sharpe(ls_mse["ls"])},
    {"rule": "Toy economic tilt", "mean_m": ls_econ["ls"].mean(), "sharpe_ann": sharpe(ls_econ["ls"])},
])
summary.round(4)

In [ ]:
cum = pd.DataFrame({
    "MSE": (1 + ls_mse["ls"]).cumprod(),
    "Toy economic tilt": (1 + ls_econ["ls"]).cumprod(),
})
ax = cum.plot(title="Cumulative long–short (sketch, not Chen estimator)")
ax.set_ylabel("Growth of $1")
plt.tight_layout()
plt.show()

## 5. Discussion (map back to the paper)

1. Gu optimizes predictive fit; Chen asks for **pricing / no-arbitrage** discipline.  
2. Adversarial test assets ≈ “don’t let the model look good on easy portfolios only.”  
3. Macro-state networks ≈ conditioning on the economy, not only firm characteristics.  
4. What would go wrong if the economic constraint is **misspecified**?

**Homework seed:** sketch (bullets) how you would put a Sharpe / SDF penalty *inside* training, not post-hoc.

## Takeaway

After the ML horse race, the frontier is **which economic restrictions belong in the objective** — Chen–Pelger–Zhu is a leading answer in asset pricing.